## Import files and libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

At the moment, two files are needed. $\mathit{raw}$ reads the file for non normalized (raw) factors, from Download Factors. $\mathit{perf}$ reads the file with the stock performance, from AI Factors. We only need the future performance from the $\mathit{perf}$ file

In [ ]:
raw=pd.read_csv('backtest_raw_1w.csv')

perf=pd.read_csv('backtest_1w_perf.csv')

Fill Nan values in the factor columns with their median value. This ensures that artificial data will be excluded when building our long-short portfolio.

NOTE: in this test file, the SP500 performance is called $\mathit{benchmark}$; this is an arbitrary name as the formula was handwritten on the website. It might be generalized in the future. The other names are universal.

In [ ]:
factors = [c for c in raw.columns.to_list() if c not in ['Date', 'P123 ID', 'Ticker', 'benchmark']]

for col in factors:
    if col in raw.columns:  # Ensure the column exists in the dataframe
        raw[col] = raw[col].fillna(raw[col].median())

Merge the Future Performance column into the raw dataframe

In [ ]:
df = raw.merge(
    perf[['Date', 'Ticker', 'Future Perf']],  # Select only necessary columns from perf
    on=['Date', 'Ticker'],  # Match based on 'Date' and 'Ticker'
    how='left'  # Use a left join to preserve all rows in new_df
).dropna()# Drop any eventual Na emerging from perf and raw having different number of rows

## Computations

Building the Long-Short portfolios.

This code is equivalent to having rank>70 as a long rule and rank<30 as a short rule. If a different percentage is needed, just change the numerical value in the definitions of top_n, bottom_n

In [ ]:
results = []
# Iterate through each column in factors
for col in factors:
    if col in df.columns:
        # Group by timestamp and perform the operation for each date
        print('Analyzing factor %s'%(col))
        for date, group in df.groupby('Date'):
            # Sort the group by the column in descending order
            group_sorted = group.sort_values(by=col, ascending=False)

            # Determine the number of rows in the top and bottom 30%
            n = len(group_sorted)
            top_n = int(n * 0.3) # change this number for different long rules
            bottom_n = int(n * 0.3) # change this number for different short rules

            # Select the top 30% and bottom 30% of the column
            top_30 = group_sorted.iloc[:top_n]
            bottom_30 = group_sorted.iloc[-bottom_n:]

            # Sum the values in the 'Future Perf' column for top and bottom 30%
            top_sum = top_30['Future Perf'].sum()/100
            bottom_sum = bottom_30['Future Perf'].sum()/100

            # Calculate the difference and store the result
            value = (top_sum - bottom_sum)/(top_n + bottom_n)
            results.append({'Date': date, 'factor': col, 'ret': value})


# Convert results to a DataFrame for better readability
results_df = pd.DataFrame(results)


Analyzing factor 1w Return
Analyzing factor 1Yr Return
Analyzing factor 3Mo Return 6M Ago
Analyzing factor Accounts Receivables Growth PYQ
Analyzing factor Asset Turnover Growth 3Y
Analyzing factor BeneishMScore
Analyzing factor Beta 5Y
Analyzing factor Cash & Equiv Growth PYQ
Analyzing factor Chaikin Money Flow
Analyzing factor Current Ratio Growth 3Y
Analyzing factor Debt LT to Eq Q
Analyzing factor Earnings Yield
Analyzing factor EBIT Growth PYQ
Analyzing factor EBITDA Yield
Analyzing factor Enterprise Value
Analyzing factor EPS Actual Growth 3Y
Analyzing factor EV to EBITDATTM
Analyzing factor Free Cash Flow Yield
Analyzing factor Gross Profit to EV
Analyzing factor Gross Profit TTM
Analyzing factor Gross Profitability Ratio Growth 3Y
Analyzing factor Industry 1Y Ret
Analyzing factor Interest Coverage TTM
Analyzing factor Long Term Debt Growth PYQ
Analyzing factor Net FCF Growth PYQ
Analyzing factor Net FCF Yield
Analyzing factor Net Profit Margin Growth TTM
Analyzing factor NetInc

Now compare the portfolios with the benchmark to compute $\beta$, $\alpha$ and their t-statistic.

In [ ]:
benchmark=df[['Date', 'benchmark']].drop_duplicates()

# Merge portfolio returns and benchmark returns on the date
merged_data = results_df.merge(benchmark, on='Date', how='inner')

# Loop through each column in the results to calculate beta and alpha
metrics = []

for col in results_df['factor'].unique():
    # Filter for the specific column
    subset = merged_data[merged_data['factor'] == col]

    # Independent variable (benchmark returns)
    x = subset['benchmark']  # Ensure benchmark returns are in decimal form
    # Dependent variable (portfolio returns)
    y = subset['ret']
    # Compute beta and alpha as a linear regression: return = alpha + beta * benchmark
    beta, alpha = np.polyfit(x, y, deg=1)
    # annualized alpha with weekly data
    ann_alpha=100*((1+alpha)**(52)-1)
    # T-student test
    t_stat, p_value = stats.ttest_1samp(y, popmean=0)  # One-sample t-test
    # Store the results
    metrics.append({'column': col, 'T Statistic':t_stat,'p-value':p_value, 'beta': beta,'alpha': alpha,'annualized alpha %':ann_alpha})

# Convert results to a DataFrame for better readability
metrics_df = pd.DataFrame(metrics)


Compute the correlation between portfolios

In [ ]:
pivot_df = results_df.pivot(index='Date', columns='factor', values='ret')

corr_matrix = pivot_df.corr()

Finally, select the best features: choose those with the maximal possible alpha and correlation below some threshold.

The minimal acceptable values for alpha and correlation thresholds can be changed when calling this function

In [ ]:
def select_best_features(alpha_beta_df, correlation_matrix, N=20, correlation_threshold=0.5, a_min=0.5):
    """
    Select the N best features based on maximal absolute alpha and minimal correlation,
    with an alpha threshold filter.

    Parameters:
        alpha_beta_df (pd.DataFrame): DataFrame with feature names and their metrics (alpha, beta, p-value...)
            Must include columns: 'column' and 'annualized alpha %'.
        correlation_matrix (pd.DataFrame): Correlation matrix of features.
        N (int): Number of features to select. Default value is 20.
        correlation_threshold (float): Maximum allowed correlation between selected features. Default value is 0.5
        a_min (float): Minimum absolute annualized alpha % value for features to be considered. Default value is 0.5%.

    Returns:
        list: List of selected feature names.
    """
    # Filter features by alpha threshold (absolute alpha >= a_min)
    filtered_alpha = alpha_beta_df[alpha_beta_df['annualized alpha %'].abs() >= a_min]

    # Sort features by absolute alpha in descending order
    sorted_alpha = filtered_alpha.sort_values(by='annualized alpha %', key=abs, ascending=False)

    # Initialize selected features
    selected_features = []

    for feature in sorted_alpha['column']:
        # Check correlation of this feature with already selected features
        if all(
            abs(correlation_matrix.loc[feature, selected]) < correlation_threshold
            for selected in selected_features
        ):
            # Add feature to selected list if correlation is below the threshold
            selected_features.append(feature)

        # Stop if we've selected N features
        if len(selected_features) >= N:
            break

    return selected_features

Example usage. You can modify the numbers on the right for different number of selected features, different thresholds for alpha and correlation.

In [ ]:
selected_features = select_best_features(
    alpha_beta_df = metrics_df,
    correlation_matrix = corr_matrix,
    N = 10,
    correlation_threshold = 0.5,
    a_min = 0.5
)

print("Selected Features:", selected_features)

Selected Features: ['Short Interest Ratio', 'Net FCF Yield', 'NetInc to Assets TTM', 'Net FCF Growth PYQ', 'Cash & Equiv Growth PYQ', '1w Return', 'Up-Down EPS Rev Net 4 weeks', 'Asset Turnover Growth 3Y']


Check the selected portfolios and their correlation

In [ ]:
metrics_df[metrics_df.column.isin(selected_features)].sort_values(by='annualized alpha %',key=abs, ascending=False)


,column,T Statistic,p-value,beta,alpha,annualized alpha %
39,Short Interest Ratio,-2.272932,0.023847,-0.026712,-0.000549,-2.813721
25,Net FCF Yield,1.532085,0.126717,0.007437,0.000493,2.597575
27,NetInc to Assets TTM,0.287208,0.774182,-0.080396,0.000422,2.219048
24,Net FCF Growth PYQ,0.726435,0.468226,-0.024253,0.000274,1.436281
7,Cash & Equiv Growth PYQ,0.889047,0.374799,-0.010015,0.000240,1.257818
0,1w Return,-0.644080,0.520092,-0.084757,-0.000202,-1.046261
43,Up-Down EPS Rev Net 4 weeks,-0.061805,0.950766,-0.061456,0.000158,0.822349
4,Asset Turnover Growth 3Y,0.467504,0.640531,0.017663,0.000114,0.592461


In [ ]:
corr_matrix.loc[selected_features, selected_features]


factor,Short Interest Ratio,Net FCF Yield,NetInc to Assets TTM,Net FCF Growth PYQ,Cash & Equiv Growth PYQ,1w Return,Up-Down EPS Rev Net 4 weeks,Asset Turnover Growth 3Y
factor,,,,,,,,
Short Interest Ratio,1.000000,-0.266467,0.166907,-0.132865,-0.135645,0.296405,0.053192,-0.201259
Net FCF Yield,-0.266467,1.000000,-0.220589,0.241410,0.097672,0.021384,-0.085088,0.124411
NetInc to Assets TTM,0.166907,-0.220589,1.000000,0.353699,-0.171868,0.109305,0.450942,-0.262993
Net FCF Growth PYQ,-0.132865,0.241410,0.353699,1.000000,-0.038404,-0.047412,0.453515,-0.319850
Cash & Equiv Growth PYQ,-0.135645,0.097672,-0.171868,-0.038404,1.000000,0.094113,0.120380,-0.002334
1w Return,0.296405,0.021384,0.109305,-0.047412,0.094113,1.000000,0.051414,-0.042436
Up-Down EPS Rev Net 4 weeks,0.053192,-0.085088,0.450942,0.453515,0.120380,0.051414,1.000000,-0.249720
Asset Turnover Growth 3Y,-0.201259,0.124411,-0.262993,-0.319850,-0.002334,-0.042436,-0.249720,1.000000
